

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 96.0 MB/s eta 0:00:00
✔ Downl

## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [3]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [4]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

Writing db.py


In [5]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [6]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [35]:
%%writefile app.py
import os, re, requests, streamlit as st
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                  verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

# --- Page Config MUST be first ---
st.set_page_config(page_title="Wellness Portal", page_icon="🌱", layout="wide")

# --- UI/UX ANIMATION & STYLING (MODERN WEB APP THEME) ---
def apply_custom_styling():
    st.markdown("""
    <style>
    /* 1. Global App Background - Slow Moving Soft Gradient */
    .stApp {
        background: linear-gradient(120deg, #fdfbfb 0%, #ebedee 100%, #fdfbfb 100%);
        background-size: 200% 200%;
        animation: flowingBackground 15s ease infinite;
    }

    @media (prefers-color-scheme: dark) {
        .stApp {
            background: linear-gradient(120deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            background-size: 200% 200%;
        }
    }

    @keyframes flowingBackground {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }

    /* 2. Hide default Streamlit top header for a cleaner app feel */
    [data-testid="stHeader"] {
        background-color: transparent !important;
    }

    /* 3. Staggered Fade & Slide up for main containers */
    .block-container {
        animation: fadeSlideUp 0.8s cubic-bezier(0.16, 1, 0.3, 1) forwards;
    }
    @keyframes fadeSlideUp {
        0% { opacity: 0; transform: translateY(30px); }
        100% { opacity: 1; transform: translateY(0); }
    }

    /* 4. Animated Gradient Text */
    .gradient-text {
        background: linear-gradient(-45deg, #FF416C, #FF4B2B, #4b6cb7, #182848);
        background-size: 300% 300%;
        animation: textGradient 5s ease infinite;
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-weight: 900;
        letter-spacing: -0.5px;
        margin-bottom: 0px;
    }
    @keyframes textGradient {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }

    /* 5. Glassmorphism Forms & Containers */
    [data-testid="stForm"], .stChatInputContainer > div {
        background: rgba(255, 255, 255, 0.05) !important;
        backdrop-filter: blur(16px) saturate(180%);
        -webkit-backdrop-filter: blur(16px) saturate(180%);
        border: 1px solid rgba(200, 200, 200, 0.2);
        border-radius: 20px;
        padding: 2rem;
        box-shadow: 0 8px 32px 0 rgba(31, 38, 135, 0.1);
        transition: transform 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275), box-shadow 0.4s ease;
    }
    [data-testid="stForm"]:hover {
        transform: translateY(-5px);
        box-shadow: 0 15px 35px 0 rgba(31, 38, 135, 0.15);
    }

    /* 6. Liquid / Bouncing Button Animations */
    div.stButton > button {
        background-size: 200% auto;
        background-image: linear-gradient(to right, #4CB8C4 0%, #3CD3AD 51%, #4CB8C4 100%);
        color: white !important;
        border: none !important;
        border-radius: 30px !important;
        font-weight: 700;
        letter-spacing: 0.5px;
        transition: 0.5s;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    div.stButton > button:hover {
        background-position: right center; /* trigger gradient shift */
        transform: scale(1.05) translateY(-2px);
        box-shadow: 0 10px 20px rgba(60, 211, 173, 0.4);
    }
    div.stButton > button:active {
        transform: scale(0.95) translateY(2px);
    }

    /* Secondary Buttons Styling Overrides (For back buttons, etc.) */
    div.stButton > button:first-child:not([type="primary"]) {
        background-image: linear-gradient(to right, #ece9e6 0%, #ffffff 51%, #ece9e6 100%);
        color: #333 !important;
        border: 1px solid rgba(0,0,0,0.1) !important;
    }
    @media (prefers-color-scheme: dark) {
        div.stButton > button:first-child:not([type="primary"]) {
            background-image: linear-gradient(to right, #2b5876 0%, #4e4376 51%, #2b5876 100%);
            color: #fff !important;
            border: 1px solid rgba(255,255,255,0.1) !important;
        }
    }

    /* 7. Chat Bubbles Modernization */
    [data-testid="stChatMessage"] {
        background-color: transparent;
        animation: chatSlideIn 0.4s cubic-bezier(0.25, 1, 0.5, 1);
        padding: 10px;
        border-radius: 15px;
    }
    [data-testid="stChatMessage"] [data-testid="stMarkdownContainer"] {
        background: rgba(120, 120, 120, 0.1);
        padding: 15px 20px;
        border-radius: 0px 20px 20px 20px;
        display: inline-block;
    }
    /* Make user messages look distinct */
    [data-testid="stChatMessage"]:nth-child(even) [data-testid="stMarkdownContainer"] {
        background: rgba(76, 184, 196, 0.15);
        border-radius: 20px 0px 20px 20px;
        border: 1px solid rgba(76, 184, 196, 0.3);
    }

    @keyframes chatSlideIn {
        0% { opacity: 0; transform: translateY(20px) scale(0.95); }
        100% { opacity: 1; transform: translateY(0) scale(1); }
    }

    /* 8. Input Fields Smooth Focus */
    .stTextInput>div>div>input {
        border-radius: 10px;
        transition: all 0.3s ease;
        border: 2px solid transparent;
        background-color: rgba(150, 150, 150, 0.08);
    }
    .stTextInput>div>div>input:focus {
        border-color: #4CB8C4 !important;
        box-shadow: 0 0 15px rgba(76, 184, 196, 0.3) !important;
        background-color: transparent;
        transform: translateY(-2px);
    }
    </style>
    """, unsafe_allow_html=True)

apply_custom_styling()

# URL of the FastAPI backend. Set BACKEND_URL as an env var / secret in deployment;
# defaults to localhost for local dev (e.g. running `uvicorn backend:app` alongside Streamlit).
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []

def goto(p): st.session_state.page = p; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

# ---- Logged-in view: file upload + NLP analysis page ----
if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        # Header Section
        head_col1, head_col2 = st.columns([5, 1])
        with head_col1:
            st.markdown("<h1 class='gradient-text'>📁 Employee Wellness Dashboard</h1>", unsafe_allow_html=True)
            st.caption(f"👋 Welcome back, **{user['username']}** ({user['email']})")
        with head_col2:
            st.write("") # Vertical alignment spacing
            if st.button("🚪 Log out", use_container_width=True):
                st.session_state.token = None; goto("login")

        st.divider()

        # Main Content Layout
        upload_col, spacer, chat_col = st.columns([1.2, 0.1, 1])

        with chat_col:
            st.subheader("💬 Wellness Chat")
            st.caption("A supportive space to talk about how you're feeling. Not a substitute for professional care.")

            chat_box = st.container(height=450)
            with chat_box:
                for turn in st.session_state.chat_history:
                    with st.chat_message(turn["role"]):
                        st.write(turn["content"])

            user_msg = st.chat_input("How are you feeling today?")
            if user_msg:
                st.session_state.chat_history.append({"role": "user", "content": user_msg})
                recent_history = st.session_state.chat_history[-10:-1]  # last 5 turns, excluding this message
                try:
                    resp = requests.post(
                        f"{BACKEND_URL}/chat",
                        json={"message": user_msg, "history": recent_history},
                        headers={"Authorization": f"Bearer {st.session_state.token}"},
                        timeout=60,
                    )
                    if resp.status_code == 200:
                        reply = resp.json()["reply"]
                    else:
                        reply = "Sorry, I couldn't reach the wellness assistant right now."
                except requests.exceptions.RequestException:
                    reply = "Sorry, I couldn't reach the wellness assistant right now."
                st.session_state.chat_history.append({"role": "assistant", "content": reply})
                st.rerun()

            if st.session_state.chat_history and st.button("🗑️ Clear chat", use_container_width=True):
                st.session_state.chat_history = []
                st.rerun()

        with upload_col:
            st.subheader("📊 NLP Data Analysis")
            uploaded = st.file_uploader("Upload employee feedback (CSV or TXT)", type=["csv", "txt"])
            headers = {"Authorization": f"Bearer {st.session_state.token}"}

            if uploaded is not None:
                is_csv = uploaded.name.lower().endswith(".csv")
                column_name = None

                with st.container():
                    if is_csv:
                        column_name = st.text_input(
                            "Feedback column name (leave blank to use the last column)",
                            placeholder="e.g. employee_comments"
                        ).strip() or None

                    c1, c2 = st.columns(2)

                    if c1.button("👀 Upload & Preview", use_container_width=True):
                        files = {"file": (uploaded.name, uploaded.getvalue())}
                        try:
                            resp = requests.post(f"{BACKEND_URL}/upload", files=files,
                                                 headers=headers, timeout=15)
                        except requests.exceptions.RequestException as e:
                            st.error(f"Could not reach backend: {e}")
                            resp = None

                        if resp is not None:
                            if resp.status_code == 200:
                                data = resp.json()
                                st.toast(f"File '{data['filename']}' loaded successfully!", icon="✅")
                                st.success(f"**Rows detected:** {data['row_count']}")

                                if data["type"] == "csv" and data["columns"]:
                                    st.markdown(f"**Columns:** `{', '.join(data['columns'])}`")
                                    if data["preview_rows"]:
                                        st.dataframe(
                                            [dict(zip(data["columns"], row)) for row in data["preview_rows"]],
                                            use_container_width=True
                                        )
                                elif data["preview_lines"]:
                                    st.text("\n".join(data["preview_lines"]))
                            else:
                                try:
                                    st.error(resp.json().get("detail", "Upload failed."))
                                except ValueError:
                                    st.error(f"Upload failed (status {resp.status_code}).")

                    if c2.button("🚀 Run NLP Analysis", use_container_width=True, type="primary"):
                        files = {"file": (uploaded.name, uploaded.getvalue())}
                        form = {"column": column_name} if column_name else {}
                        with st.spinner("🤖 Running multilingual NLP pipeline..."):
                            try:
                                resp = requests.post(f"{BACKEND_URL}/analyze", files=files, data=form,
                                                     headers=headers, timeout=120)
                            except requests.exceptions.RequestException as e:
                                st.error(f"Could not reach backend: {e}")
                                resp = None

                        if resp is not None:
                            if resp.status_code != 200:
                                try:
                                    st.error(resp.json().get("detail", "Analysis failed."))
                                except ValueError:
                                    st.error(f"Analysis failed (status {resp.status_code}).")
                            else:
                                r = resp.json()
                                st.toast("Analysis Complete!", icon="🎉")
                                st.markdown("### 📋 Analysis Report")

                                st.info(f"**File:** `{r['filename']}` ({r['file_type']}) | **Detected Language:** {r['detected_language']} ({r['language_code']})")
                                if r.get("used_column"):
                                    st.caption(f"Target Column: `{r['used_column']}`")

                                t1, t2 = st.tabs(["Metrics & Sentiment", "Text Breakdown"])

                                with t1:
                                    scores = r["sentiment_scores"]
                                    st.markdown(f"#### Sentiment: **{r['final_sentiment'].upper()}**")
                                    st.progress(max(0.0, min(1.0, (scores['compound'] + 1) / 2)),
                                                text=f"Compound Score: {scores['compound']:.3f}")
                                    st.bar_chart({
                                        "Positive": scores["pos"],
                                        "Negative": scores["neg"],
                                        "Neutral": scores["neu"],
                                    }, height=200)

                                    st.markdown(f"#### Emotion: **{r['final_emotion'].upper()}**")
                                    st.bar_chart(r["emotion_scores"], height=200)

                                    st.markdown("#### Processing Statistics")
                                    m1, m2, m3 = st.columns(3)
                                    m1.metric("Characters", r["original_char_count"])
                                    m2.metric("Sentences", len(r["sentences"]))
                                    m3.metric("Emojis Found", len(r["emoji_list"]))

                                    m4, m5, m6 = st.columns(3)
                                    m4.metric("Original Tokens", len(r["original_tokens"]))
                                    m5.metric("Filtered Tokens", len(r["filtered_tokens"]))

                                    if r["emoji_list"]:
                                        st.write("**Extracted Emojis:**", " ".join(r["emoji_list"]))

                                with t2:
                                    with st.expander("🧹 Cleaned Text"):
                                        st.write(r["cleaned_text"])
                                    with st.expander("📝 Sentences"):
                                        for i, s in enumerate(r["sentences"], 1):
                                            st.markdown(f"**{i}.** {s}")
                                    with st.expander("🧩 Tokens"):
                                        st.write("**Original:**", r["original_tokens"])
                                        st.write("**Filtered:**", r["filtered_tokens"])
                                    with st.expander("🌐 Translation & Lemmatization"):
                                        st.write("**English Translation:**", r["translated_text"])
                                        st.write("**Lemmatized:**", r["lemmatized_text"])

        st.stop()
    st.session_state.token = None

# ---- Helper for Centered Authentication Layout ----
def get_centered_column():
    _, center_col, _ = st.columns([1, 1.5, 1])
    return center_col

# ---- Login ----
if st.session_state.page == "login":
    with get_centered_column():
        st.markdown("<h1 style='text-align: center;' class='gradient-text'>🔐 Login</h1>", unsafe_allow_html=True)
        st.write("") # spacing
        with st.form("login"):
            email = st.text_input("Email", placeholder="you@company.com")
            pw = st.text_input("Password", type="password", placeholder="••••••••")
            go = st.form_submit_button("Log in", use_container_width=True)

        if go:
            user = get_user(email.strip().lower())
            if not user or not check_pw(pw, user["password_hash"]):
                st.error("Invalid email or password.")
            elif not user["is_verified"]:
                st.warning("Verify your email first.")
                st.session_state.email = user["email"]; goto("verify")
            else:
                st.toast("Login successful!", icon="🔓")
                st.session_state.token = make_token(user); st.rerun()

        c1, c2 = st.columns(2)
        if c1.button("✨ Create Account", use_container_width=True): goto("signup")
        if c2.button("🔑 Forgot Password?", use_container_width=True): goto("forgot")

# ---- Signup ----
elif st.session_state.page == "signup":
    with get_centered_column():
        st.markdown("<h1 style='text-align: center;' class='gradient-text'>✨ Sign Up</h1>", unsafe_allow_html=True)
        st.write("")
        with st.form("signup"):
            username = st.text_input("Username", placeholder="e.g. JohnDoe")
            email = st.text_input("Email", placeholder="you@company.com")
            pw = st.text_input("Password", type="password", help="8+ characters, including letters and numbers")
            go = st.form_submit_button("Create Account", use_container_width=True)

        if go:
            email = email.strip().lower()
            if len(username) < 3:
                st.error("Username too short.")
            elif not valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif username_taken(username) or get_user(email):
                st.error("Username or email already in use.")
            else:
                create_user(username, email, pw)
                code = new_otp(); save_otp(email, code, "signup")
                ok, msg = send_otp(email, code, "signup")
                if ok:
                    st.session_state.email = email
                    st.success("Check your email for the verification code.")
                    goto("verify")
                else:
                    st.error(f"Email failed: {msg}")

        if st.button("← Back to Login", use_container_width=True): goto("login")

# ---- Verify OTP (signup) ----
elif st.session_state.page == "verify":
    with get_centered_column():
        st.markdown("<h1 style='text-align: center;' class='gradient-text'>🛡️ Verify</h1>", unsafe_allow_html=True)
        email = st.session_state.email
        st.info(f"Enter the 6-digit code sent to **{email}**")
        with st.form("verify"):
            code = st.text_input("Verification Code", max_chars=6, placeholder="123456")
            go = st.form_submit_button("Verify Email", use_container_width=True)

        if go:
            if check_otp(email, code.strip(), "signup"):
                verify_user(email)
                st.success("Account verified! Please log in.")
                goto("login")
            else:
                st.error("Invalid or expired code.")

        if st.button("← Back to Login", use_container_width=True): goto("login")

# ---- Forgot password ----
elif st.session_state.page == "forgot":
    with get_centered_column():
        st.markdown("<h1 style='text-align: center;' class='gradient-text'>🔑 Reset</h1>", unsafe_allow_html=True)
        st.write("")
        with st.form("forgot"):
            email = st.text_input("Your account email", placeholder="you@company.com")
            go = st.form_submit_button("Send Reset Code", use_container_width=True)

        if go:
            email = email.strip().lower()
            if get_user(email):
                code = new_otp(); save_otp(email, code, "password_reset")
                send_otp(email, code, "password_reset")
            st.session_state.email = email
            st.info("If that email exists, a code has been sent.")
            goto("reset")

        if st.button("← Back to Login", use_container_width=True): goto("login")

# ---- Reset password ----
elif st.session_state.page == "reset":
    with get_centered_column():
        st.markdown("<h1 style='text-align: center;' class='gradient-text'>🔒 New Password</h1>", unsafe_allow_html=True)
        email = st.session_state.email
        with st.form("reset"):
            code = st.text_input("Reset code", max_chars=6, placeholder="123456")
            pw = st.text_input("New password", type="password", placeholder="••••••••")
            go = st.form_submit_button("Reset Password", use_container_width=True)

        if go:
            if not valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif not check_otp(email, code.strip(), "password_reset"):
                st.error("Invalid or expired code.")
            else:
                set_password(email, pw)
                st.success("Password successfully reset. Please log in.")
                goto("login")

        if st.button("← Back to Login", use_container_width=True): goto("login")

Overwriting app.py


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [27]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (Qwen LLM).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, Qwen model) load once at import
time via lazy module-level globals, so repeated /analyze calls reuse them.
"""

import re
import json as _json
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import AutoModelForCausalLM, AutoTokenizer
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Human-readable names for common languages this pipeline is likely to see.
# Purely cosmetic (shown in the UI) -- falls back to the raw ISO code if a
# language isn't listed here, so it never blocks processing of any language.
LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

# Fixed label set the Qwen prompt is constrained to. Keeping this list stable
# means downstream code (backend.py, Streamlit charts) never has to change,
# no matter which underlying model produces the emotion.
EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-1.5B-Instruct once per process (GPU if available)."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _qwen_emotion(text: str) -> dict:
    """
    Prompts Qwen to classify `text` into one of EMOTION_LABELS and return a
    confidence score per label as strict JSON. Falls back to Neutral if the
    model output can't be parsed (LLMs don't always obey format instructions).
    """
    model, tokenizer = _get_qwen()

    labels_str = ", ".join(EMOTION_LABELS)
    system_prompt = (
        "You are an emotion classification engine for employee wellness feedback. "
        f"Classify the given text into exactly one of these emotions: {labels_str}. "
        "Respond with ONLY a JSON object, no other text, in this exact format: "
        '{"emotion": "<one label>", "scores": {"Happy": 0-1, "Sad": 0-1, '
        '"Stress": 0-1, "Angry": 0-1, "Fear": 0-1, "Neutral": 0-1}}'
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text if text.strip() else "(empty feedback)"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    try:
        # Model may wrap JSON in markdown fences; strip those defensively.
        reply_clean = reply.replace("```json", "").replace("```", "").strip()
        parsed = _json.loads(reply_clean)
        emotion = parsed.get("emotion", "Neutral")
        scores = parsed.get("scores", {})
        if emotion not in EMOTION_LABELS:
            emotion = "Neutral"
        scores = {label: float(scores.get(label, 0.0)) for label in EMOTION_LABELS}
    except Exception:
        emotion, scores = "Neutral", {label: 0.0 for label in EMOTION_LABELS}
        scores["Neutral"] = 1.0

    return {"emotion": emotion, "scores": scores}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"

    # --- Emotion detection via Qwen LLM (replaces old keyword matching) ---
    qwen_result = _qwen_emotion(translated_text)
    emotion_scores = qwen_result["scores"]
    final_emotion_label = qwen_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }


# --- Crisis-keyword safety net for the wellness chatbot -----------------
# This is a simple, deliberately blunt keyword check that runs regardless
# of what the LLM says. If it fires, we always show crisis resources —
# we never rely on the small LLM alone to catch something this important.
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the same Qwen model
    already loaded for emotion detection.

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}


Overwriting nlp_pipeline.py


In [28]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)  # exclude header from data row count

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1  # default to last column

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)  # just verifies the token; raises 401 if invalid

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results


class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)  # verifies the token; raises 401 if invalid

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Overwriting backend.py


In [10]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [29]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://flashback-virtuous-upheld.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")